# 00 - Download e Organizacao do Dataset NIH Chest X-rays

Este notebook documenta o processo de download, organizacao e validacao inicial do dataset publico NIH Chest X-rays, recomendado no enunciado do PBL.

Dataset: https://www.kaggle.com/datasets/nih-chest-xrays/data

Objetivos deste notebook:

1. Baixar o dataset pelo Kaggle, preferencialmente com `kagglehub`.
2. Organizar o arquivo de metadados em `data/raw/Data_entry_2017.csv`.
3. Organizar as imagens em `data/raw/images/`.
4. Criar o indice local `data/raw/image_paths.csv`.
5. Validar se as imagens citadas no CSV existem localmente.
6. Visualizar uma pequena amostra de imagens.

Este projeto tem finalidade exclusivamente academica e educacional. Ele nao deve ser usado como ferramenta diagnostica real.

## Pre-requisitos

Antes de executar o download:

1. Crie uma conta no Kaggle.
2. Acesse a pagina do dataset e aceite os termos, se a plataforma solicitar.
3. Configure uma das formas de autenticacao:

- Arquivo `~/.kaggle/kaggle.json`.
- Variaveis de ambiente `KAGGLE_USERNAME` e `KAGGLE_KEY`.
- Sessao autenticada compativel com `kagglehub`.

Nunca suba `kaggle.json` para o GitHub. O `.gitignore` deste projeto ja ignora esse arquivo.

In [ ]:
# Dependencias opcionais para download.
# Se estiver em um ambiente novo, altere INSTALL_MISSING para True e execute esta celula.

INSTALL_MISSING = False

if INSTALL_MISSING:
    import sys
    !{sys.executable} -m pip install -q kagglehub kaggle pandas pillow matplotlib
else:
    print("Instalacao automatica desativada. Se faltar dependencia, instale: kagglehub kaggle pandas pillow matplotlib")

In [ ]:
from __future__ import annotations

import os
import shutil
import sys
import zipfile
from pathlib import Path

import pandas as pd
from PIL import Image

# Detecta a raiz do projeto tanto quando o notebook roda em notebooks/ quanto na raiz.
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "src").exists() else CURRENT_DIR.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config

config.ensure_project_directories()

RAW_DIR = config.RAW_DATA_DIR
IMAGES_DIR = config.RAW_IMAGES_DIR
DOWNLOAD_DIR = RAW_DIR / "_kaggle_download"
CSV_TARGET = RAW_DIR / config.DATA_ENTRY_FILENAME
IMAGE_INDEX_TARGET = RAW_DIR / config.IMAGE_PATHS_FILENAME

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Dataset:", config.DATASET_SLUG)
print("RAW_DIR:", RAW_DIR)
print("IMAGES_DIR:", IMAGES_DIR)
print("CSV_TARGET:", CSV_TARGET)

## Checagem de Autenticacao do Kaggle

Esta checagem nao imprime credenciais. Ela apenas indica se existe alguma forma comum de autenticacao disponivel.

In [ ]:
kaggle_json_candidates = [
    Path.home() / ".kaggle" / "kaggle.json",
    PROJECT_ROOT / "kaggle.json",
]

auth_status = {
    "has_kaggle_username_env": bool(os.environ.get("KAGGLE_USERNAME")),
    "has_kaggle_key_env": bool(os.environ.get("KAGGLE_KEY")),
    "kaggle_json_locations_found": [str(path) for path in kaggle_json_candidates if path.exists()],
}

auth_status

## Download do Dataset

A rotina abaixo tenta, nesta ordem:

1. Reutilizar arquivos locais se o CSV e as imagens ja existirem.
2. Baixar com `kagglehub.dataset_download("nih-chest-xrays/data")`.
3. Baixar com a Kaggle API tradicional.

Se ambas as opcoes falharem, use o fallback manual descrito no fim do notebook.

In [ ]:
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg"}


def count_local_images(images_dir: Path) -> int:
    return sum(1 for path in images_dir.rglob("*") if path.suffix.lower() in IMAGE_EXTENSIONS)


def local_dataset_ready() -> bool:
    return CSV_TARGET.exists() and count_local_images(IMAGES_DIR) > 0


dataset_source_dir: Path | None = None

if local_dataset_ready():
    print("Dataset local encontrado. Download nao sera repetido.")
    dataset_source_dir = RAW_DIR
else:
    try:
        import kagglehub

        downloaded_path = kagglehub.dataset_download(config.DATASET_SLUG)
        dataset_source_dir = Path(downloaded_path).resolve()
        print("Download via kagglehub concluido em:", dataset_source_dir)
    except Exception as kagglehub_error:
        print("kagglehub nao conseguiu baixar o dataset.")
        print(type(kagglehub_error).__name__, kagglehub_error)
        print("Tentando Kaggle API tradicional...")

        try:
            from kaggle.api.kaggle_api_extended import KaggleApi

            api = KaggleApi()
            api.authenticate()
            api.dataset_download_files(
                config.DATASET_SLUG,
                path=str(DOWNLOAD_DIR),
                unzip=True,
                quiet=False,
            )
            dataset_source_dir = DOWNLOAD_DIR.resolve()
            print("Download via Kaggle API concluido em:", dataset_source_dir)
        except Exception as kaggle_api_error:
            print("Kaggle API tambem falhou.")
            print(type(kaggle_api_error).__name__, kaggle_api_error)
            print("Siga o fallback manual no final do notebook.")

dataset_source_dir

## Organizacao do CSV de Metadados

O arquivo de metadados pode aparecer com pequenas variacoes no nome, como `Data_entry_2017.csv` ou `Data_Entry_2017.csv`. A rotina abaixo localiza o CSV e padroniza uma copia em `data/raw/Data_entry_2017.csv`.

In [ ]:
def find_data_entry_csv(search_roots: list[Path]) -> Path | None:
    candidate_names = {
        "data_entry_2017.csv",
        "data_entry_2017_v2020.csv",
        "dataentry2017.csv",
    }
    for root in search_roots:
        if not root.exists():
            continue
        for path in root.rglob("*.csv"):
            normalized = path.name.lower().replace("-", "_")
            if normalized in candidate_names or "data_entry" in normalized:
                return path
    return None


search_roots = [path for path in [dataset_source_dir, DOWNLOAD_DIR, RAW_DIR] if path is not None]
csv_source = find_data_entry_csv(search_roots)

if csv_source is None and CSV_TARGET.exists():
    csv_source = CSV_TARGET

if csv_source is None:
    raise FileNotFoundError(
        "Nao foi possivel localizar Data_entry_2017.csv. "
        "Baixe manualmente pelo Kaggle e coloque em data/raw/Data_entry_2017.csv."
    )

if csv_source.resolve() != CSV_TARGET.resolve():
    shutil.copy2(csv_source, CSV_TARGET)

df_metadata = pd.read_csv(CSV_TARGET)

print("CSV fonte:", csv_source)
print("CSV padronizado:", CSV_TARGET)
print("Linhas no CSV:", len(df_metadata))
df_metadata.head()

## Extracao e Organizacao das Imagens

No Kaggle, o NIH Chest X-rays pode vir em arquivos `.zip`, como `images_001.zip`, `images_002.zip` etc. Esta celula extrai os zips para `data/raw/images/`. Se as imagens ja estiverem extraidas, ela apenas segue para o mapeamento.

In [ ]:
def find_zip_files(search_roots: list[Path]) -> list[Path]:
    zip_files: list[Path] = []
    for root in search_roots:
        if root.exists():
            zip_files.extend(root.rglob("*.zip"))
    return sorted(set(zip_files))


def extract_zip_files(zip_files: list[Path], target_dir: Path) -> None:
    for zip_path in zip_files:
        print(f"Extraindo {zip_path.name}...")
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(target_dir)


def find_image_files(search_roots: list[Path]) -> list[Path]:
    image_files: list[Path] = []
    for root in search_roots:
        if root.exists():
            image_files.extend(path for path in root.rglob("*") if path.suffix.lower() in IMAGE_EXTENSIONS)
    return sorted(set(image_files))


def copy_extracted_images_to_raw(source_images: list[Path], target_dir: Path) -> int:
    copied = 0
    for source_path in source_images:
        if target_dir in source_path.parents:
            continue
        target_path = target_dir / source_path.name
        if not target_path.exists():
            shutil.copy2(source_path, target_path)
            copied += 1
    return copied


existing_image_count = count_local_images(IMAGES_DIR)
print("Imagens ja existentes em data/raw/images antes da extracao:", existing_image_count)

zip_files = find_zip_files(search_roots)
print("Arquivos zip encontrados:", len(zip_files))

if existing_image_count == 0 and zip_files:
    extract_zip_files(zip_files, IMAGES_DIR)
elif existing_image_count > 0:
    print("Extracao ignorada porque ja existem imagens em data/raw/images/.")
else:
    source_images = find_image_files(search_roots)
    copied_count = copy_extracted_images_to_raw(source_images, IMAGES_DIR)
    print("Nenhum zip encontrado. Imagens ja extraidas copiadas para data/raw/images/:", copied_count)

print("Imagens em data/raw/images depois da extracao:", count_local_images(IMAGES_DIR))

## Criacao do Indice Local de Imagens

O arquivo `image_paths.csv` conecta o nome da imagem usado no CSV de metadados ao caminho local no projeto. Esse indice evita ficar fazendo buscas custosas em todos os treinamentos.

In [ ]:
def build_image_index(search_dirs: list[Path]) -> pd.DataFrame:
    records = []
    seen_names: set[str] = set()

    for directory in search_dirs:
        if not directory.exists():
            continue

        for path in directory.rglob("*"):
            if path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue

            if path.name in seen_names:
                continue

            seen_names.add(path.name)
            records.append(
                {
                    "image_name": path.name,
                    "relative_path": str(path.resolve().relative_to(PROJECT_ROOT)),
                    "absolute_path": str(path.resolve()),
                    "file_size_bytes": path.stat().st_size,
                    "exists": path.exists(),
                }
            )

    return pd.DataFrame(records).sort_values("image_name").reset_index(drop=True)


image_search_dirs = [IMAGES_DIR]
if dataset_source_dir is not None and dataset_source_dir != RAW_DIR:
    image_search_dirs.append(dataset_source_dir)

image_index = build_image_index(image_search_dirs)
image_index.to_csv(IMAGE_INDEX_TARGET, index=False)

print("Indice salvo em:", IMAGE_INDEX_TARGET)
print("Imagens indexadas:", len(image_index))
image_index.head()

## Validacao Cruzada Entre CSV e Imagens

Esta etapa verifica se as imagens referenciadas em `Data_entry_2017.csv` foram encontradas localmente.

In [ ]:
required_columns = {"Image Index", "Finding Labels", "Patient ID"}
missing_columns = required_columns - set(df_metadata.columns)
if missing_columns:
    raise ValueError(f"Colunas obrigatorias ausentes no CSV: {sorted(missing_columns)}")

metadata_images = set(df_metadata["Image Index"].astype(str))
indexed_images = set(image_index["image_name"].astype(str)) if not image_index.empty else set()

found_images = metadata_images & indexed_images
missing_images = sorted(metadata_images - indexed_images)
extra_images = sorted(indexed_images - metadata_images)

validation_summary = {
    "metadata_rows": int(len(df_metadata)),
    "unique_images_in_metadata": int(len(metadata_images)),
    "indexed_local_images": int(len(indexed_images)),
    "images_found_from_metadata": int(len(found_images)),
    "images_missing_from_metadata": int(len(missing_images)),
    "extra_local_images_not_in_metadata": int(len(extra_images)),
}

validation_summary

In [ ]:
if missing_images:
    print("Primeiras imagens ausentes:")
    print(missing_images[:20])
else:
    print("Todas as imagens do CSV foram encontradas no indice local.")

if extra_images:
    print("Primeiras imagens locais extras nao referenciadas no CSV:")
    print(extra_images[:20])

## Visualizacao de Amostra

A visualizacao abaixo confirma que as imagens podem ser abertas pelo caminho final registrado no indice local.

In [ ]:
import matplotlib.pyplot as plt

sample_names = list(sorted(found_images))[:6]
path_lookup = dict(zip(image_index["image_name"], image_index["absolute_path"]))

if not sample_names:
    print("Nenhuma imagem encontrada para visualizacao. Confira download, extracao e caminhos.")
else:
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    axes = axes.ravel()

    for ax, image_name in zip(axes, sample_names):
        image_path = Path(path_lookup[image_name])
        image = Image.open(image_path).convert("L")
        label = df_metadata.loc[df_metadata["Image Index"] == image_name, "Finding Labels"].iloc[0]
        ax.imshow(image, cmap="gray")
        ax.set_title(f"{image_name}\n{label}", fontsize=9)
        ax.axis("off")

    for ax in axes[len(sample_names):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## Fallback Manual

Se o download automatico falhar por credenciais, rede ou aceite de termos no Kaggle, siga estes passos:

1. Acesse https://www.kaggle.com/datasets/nih-chest-xrays/data.
2. Baixe os arquivos do dataset.
3. Coloque o CSV de metadados em:

```text
data/raw/Data_entry_2017.csv
```

4. Extraia ou copie as imagens para:

```text
data/raw/images/
```

5. Reexecute as celulas a partir de "Organizacao do CSV de Metadados".

Criterio de sucesso: a validacao deve mostrar imagens encontradas e a amostra visual deve abrir corretamente.

## Proxima Etapa

Depois que este notebook validar o dataset local, seguir para:

`notebooks/01_eda_dataset.ipynb`

Essa proxima etapa fara a analise exploratoria: distribuicao de labels, `No Finding`, `Cardiomegaly`, casos multi-label, pacientes, idade, sexo e posicao da imagem.